In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [8]:
# See sheet names
xls = pd.ExcelFile('Online retail.xlsx')
print(xls.sheet_names)

['Sheet1']


In [9]:
# load dataset
df = pd.read_excel("Online retail.xlsx")

In [10]:
df.head()

,"shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil"
0,"burgers,meatballs,eggs"
1,chutney
2,"turkey,avocado"
3,"mineral water,milk,energy bar,whole wheat rice..."
4,low fat yogurt


In [11]:
df.size

7500

## Data Preprocessing:

In [12]:
# missing values:
df.isnull().sum()

shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil    0
dtype: int64

In [16]:
# handling duplicate
df.duplicated().sum()

np.int64(2325)

In [18]:
df.drop_duplicates(inplace = True , ignore_index = True)

In [19]:
df.duplicated().sum()

np.int64(0)

In [24]:
df.describe

<bound method NDFrame.describe of      shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
0                                burgers,meatballs,eggs                                                                                                                                                                             
1                                               chutney                                                                                                                                                                             
2                                        turkey,avocado                                                                                                                                                                             
3     mineral water,milk,energy bar,whole wheat ri

In [30]:
# Converts the dataset into list-of-lists
        # format required for Association Rule Mining.
transactions = []

for i in range(len(df)):
    transactions.append([str(df.iloc[i,j]) for j in range(len(df.columns))])

In [31]:
# one long list
        # To collect all individual items.
items1 = [items for trans in transactions for items in trans]
# Get Unique Items
        # To know how many different products exist in the dataset.
items2 = list(set(item1))

In [32]:
# Remove 'Nan'
if 'nan' in items2:
    items2.remove('nan')

In [33]:
len(items2)

5175

In [34]:
len(items1)

5175

In [35]:
print(items2)

['ground beef,mineral water,cottage cheese,magazines', 'vegetables mix,eggs,whole wheat rice', 'shrimp,whole wheat pasta,soup,milk,french fries', 'burgers,french fries,frozen smoothie,hot dogs,strawberries,green tea', 'spaghetti,whole wheat rice,corn,champagne', 'herb & pepper,ground beef,spaghetti,chocolate,milk,eggs,light cream', 'fresh tuna,herb & pepper,mineral water,olive oil,chicken,muffins,chocolate,yogurt cake', 'pepper,spaghetti,mineral water,milk,pancakes,cake,white wine,green tea', 'burgers,spaghetti,mineral water,almonds,honey,extra dark chocolate,carrots,mashed potato', 'eggs,tea', 'burgers,mineral water', 'green tea,mushroom cream sauce', 'red wine,mineral water,champagne,mushroom cream sauce', 'frozen smoothie,brownies', 'mineral water,soup,gums,extra dark chocolate,french fries', 'frozen vegetables,spaghetti,mineral water,avocado,chocolate,french fries,cookies,melons', 'pickles,herb & pepper,shrimp,milk,spinach,cake,chili,green tea,light mayo', 'frozen vegetables,spaghe

In [36]:
df.duplicated().sum()

np.int64(0)

In [40]:
print(transactions[:5])

[['burgers,meatballs,eggs'], ['chutney'], ['turkey,avocado'], ['mineral water,milk,energy bar,whole wheat rice,green tea'], ['low fat yogurt']]


In [41]:
# splits each transaction’s comma-separated string into a list of individual items
transactions = [t[0].split(',') for t in transactions]

In [43]:
## install mlxtend
!pip install mlxtend


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [45]:
# importing Transaction,apriori and association_rules
from mlxtend.frequent_patterns import apriori,association_rules
from mlxtend.preprocessing import TransactionEncoder

In [46]:
data = TransactionEncoder()

In [48]:
data_ary = data.fit(transactions).transform(transactions)
data1 = pd.DataFrame(data_ary, columns = data.columns_)

In [49]:
data1.head()

,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [53]:
# items often bought together
freq_patterns = apriori(data1, min_support=0.0005, use_colnames=True)


In [55]:
# top frequent itemsets
freq_patterns.sort_values('support', ascending=False).head()

,support,itemsets
71,0.299710,(mineral water)
99,0.229565,(spaghetti)
36,0.208116,(eggs)
24,0.205217,(chocolate)
42,0.192657,(french fries)


In [57]:
# only itemsets with more than one item
freq_patterns[freq_patterns['itemsets'].apply(lambda x: len(x) > 1)]

,support,itemsets
119,0.002319,"(avocado, almonds)"
120,0.000773,"(almonds, barbecue sauce)"
121,0.001353,"(almonds, black tea)"
122,0.000966,"(almonds, brownies)"
123,0.000773,"(almonds, bug spray)"
...,...,...
36118,0.000580,"(milk, turkey, vegetables mix, frozen vegetabl..."
36119,0.000580,"(milk, turkey, vegetables mix, frozen vegetabl..."
36120,0.000580,"(turkey, vegetables mix, frozen vegetables, wh..."
36121,0.000580,"(milk, turkey, vegetables mix, whole wheat ric..."


In [59]:
# Generates association rules
rules = association_rules(freq_patterns,metric='confidence',min_threshold=0.1)
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(almonds),(burgers),0.029179,0.113816,0.007536,0.258278,2.269252,1.0,0.004215,1.194765,0.576137,0.055635,0.163016,0.162246
1,(almonds),(cake),0.029179,0.103575,0.004444,0.152318,1.470606,1.0,0.001422,1.057502,0.329626,0.034639,0.054375,0.097614
2,(almonds),(chicken),0.029179,0.083865,0.003478,0.119205,1.421400,1.0,0.001031,1.040123,0.305379,0.031746,0.038576,0.080340
3,(almonds),(chocolate),0.029179,0.205217,0.008696,0.298013,1.452183,1.0,0.002708,1.132190,0.320740,0.038527,0.116756,0.170193
4,(almonds),(eggs),0.029179,0.208116,0.009469,0.324503,1.559243,1.0,0.003396,1.172299,0.369443,0.041561,0.146975,0.185000


In [60]:
print(data1.columns[:20])

Index([' asparagus', 'almonds', 'antioxydant juice', 'asparagus', 'avocado',
       'babies food', 'bacon', 'barbecue sauce', 'black tea', 'blueberries',
       'body spray', 'bramble', 'brownies', 'bug spray', 'burger sauce',
       'burgers', 'butter', 'cake', 'candy bars', 'carrots'],
      dtype='object')


In [61]:
print(len(transactions))

5175


# Analysis and Interpretation:

The rules show which items are commonly bought together. For example, almonds are frequently purchased with burgers, cake, chocolate, and eggs. A higher confidence indicates that if one item is bought, the other is likely to be bought too, while a lift > 1 confirms a strong positive association between the items.

Customers who buy almonds tend to also buy chocolate and eggs, suggesting these may be part of regular meal or snack combinations.

Products with high lift can be placed near each other in stores or suggested together in online carts.


Interview Questions

1. What is lift and why is it important in Association rules?

    Lift is a metric used in association rule mining to measure how much more often two items occur together than would be expected if they were independent.
    Lift helps reveal meaningful, non-random relationships between items. Even if confidence is high, a rule is only useful when lift > 1, showing the items are bought together more often than expected. This makes lift valuable for decisions like product bundling and recommendations.


2. What is support and Confidence. How do you calculate them?


   1. Support
      Support shows how frequently an item or itemset appears in the dataset.
      -dividing the number of transactions containing the itemset by the total number of transactions
  
      

   3. Confidence
    Confidence measures how likely item B is bought when item A is bought.
       -dividing the support of the combined itemset (A and B) by the support of the antecedent (A)
   

3. What are some limitations or challenges of Association rules mining?

    * It can produce too many rules, making interpretation difficult.
    * Requires careful selection of support and confidence to get meaningful rules.
    * Does not consider item order or time, only co-occurrence.
    * Can be computationally expensive on large datasets.
    